## ADE20K MSPE fine-tuning (frozen backbone, MSPE kernels only) - recipe R2

ADE20K Dataset:
https://www.kaggle.com/datasets/awsaf49/ade20k-dataset/data


In [ ]:
!python -c "import monai" || pip install -q "monai[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
!python -c "import wandb" || pip install -q wandb
%matplotlib inline

In [ ]:
import os
import sys

colab = True

if colab:
    from google.colab import drive, runtime, userdata
    drive.mount('/content/drive')
    dataset_dir = r"/content/drive/MyDrive/DATASETS/ADE20K/"  # overridden by cell below
    root_dir = r"/content/drive/MyDrive/RUNS/ADE20Resized/FINE_TUNING/"
    baseline_dir = r"/content/drive/MyDrive/MSPE/NOTEBOOKS/ADE20K/FULL_TRAINING/"
    mspe_code_dir = r"/content/drive/MyDrive/MSPE/"
    num_workers = os.cpu_count()
    batch_size = 48  # G4 instance
    cache_rate = 1.0
    DEBUG_LIMIT = None
else:
    dataset_dir = r""
    root_dir = r""
    baseline_dir = r""
    mspe_code_dir = r""
    num_workers = 0
    cache_rate = 0.0
    batch_size = 2    
    DEBUG_LIMIT = 1000  
    userdata = None

sys.path.append(mspe_code_dir)
os.makedirs(root_dir, exist_ok=True)


In [ ]:
# stage the dataset on local SSD
if colab:
    import time

    DRIVE_ADE = "/content/drive/MyDrive/DATASETS/ADE20K"       
    DRIVE_TAR = "/content/drive/MyDrive/DATASETS/ADE20K.tar"   # built once, reused later 
    LOCAL_ADE = "/content/ADE20K"

    if not os.path.isdir(LOCAL_ADE):
        t0 = time.time()
        if not os.path.isfile(DRIVE_TAR):
            print("one-time: packing dataset into a single tar on Drive (~20-40 min)...")
            !tar -cf "{DRIVE_TAR}" -C "{os.path.dirname(DRIVE_ADE)}" "{os.path.basename(DRIVE_ADE)}"
        !tar -xf "{DRIVE_TAR}" -C /content
        print(f"dataset staged locally in {time.time() - t0:.0f} s")

    dataset_dir = LOCAL_ADE + "/"

In [ ]:
import glob
import gc
import os
import time
import copy

import numpy as np
import monai
import wandb
import tqdm
import matplotlib.pyplot as plt
from PIL import Image

from monai.config import print_config
from monai.data import CacheDataset, list_data_collate, PILReader, ThreadDataLoader
from monai.data.utils import partition_dataset
from monai.inferers import sliding_window_inference
from monai.networks.blocks import PatchEmbed, UnetrUpBlock
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped, MapTransform,
    RandFlipd, RandScaleIntensityd, RandShiftIntensityd, RandAdjustContrastd, RandZoomd,
    DivisiblePadd, SpatialPadd, Resized, RandSpatialCropd,
    ScaleIntensityRanged, NormalizeIntensityd, CastToTyped,
)
from monai.utils import first, set_determinism

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from swin_mspe import (
    MSPEPatchEmbedSwinNaiveRouting,
    MSPEPatchEmbedSwinOverlapping,
    MSPEPatchEmbedSwinDilatingK3,
    mspe_swin_forward,
    img_resize,
    label_resize,
    pi_resize,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bfloat16 on Ampere and later for autocast
AMP_DTYPE = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"device: {device}, AMP dtype: {AMP_DTYPE}")

print_config()

In [ ]:
# Prepare dataset

def build_ade20k_split(split):
    image_dir = os.path.join(dataset_dir, "images", split)
    annotation_dir = os.path.join(dataset_dir, "annotations", split)
    images_list = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))

    dataset = []
    for image_path in tqdm.tqdm(images_list, desc=f"Indexing ADE20K {split}"):
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        annotation_path = os.path.join(annotation_dir, image_id + ".png")
        dataset.append({"image": image_path, "label": annotation_path})
    return dataset

train_dataset = build_ade20k_split("training")
validation_dataset = build_ade20k_split("validation")

# Kaggle dataset copy is missing the test set, so use only valdiation
val_dataset, test_dataset = partition_dataset(validation_dataset, ratios=[1.0, 0.0])

# Local smoke-test
if not colab and DEBUG_LIMIT is not None:
    train_dataset = train_dataset[:DEBUG_LIMIT]
    val_dataset = val_dataset[:16]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# Labels loading
class LoadADE20KLabeld(MapTransform):
    def __init__(self, keys="label", allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            with Image.open(d[key]) as image:
                d[key] = np.array(image, dtype=np.uint8)[None, ...]  
        return d

In [ ]:
# Dataloaders 

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

BASE_LONGEST = 683
CROP_SIZE = 512
SCALE_JITTER = (0.5, 2.0)

RESIZE_LONGEST = 512

PATCH_SIZE = 4
STRIDE_DIVISOR = PATCH_SIZE * 16

train_transforms = Compose([
    # deterministic part
    LoadImaged(
        keys=["image"],
        reader=PILReader(converter=lambda image: image.convert("RGB"), reverse_indexing=False),
    ),
    LoadADE20KLabeld(keys="label"),
    EnsureChannelFirstd(keys=["image"]),
    Resized(keys=["image", "label"], spatial_size=BASE_LONGEST, size_mode="longest", mode=("bilinear", "nearest")),
    CastToTyped(keys=["image", "label"], dtype=(torch.uint8, torch.uint8)),

    # random part
    RandZoomd(keys=["image", "label"], prob=1.0, min_zoom=SCALE_JITTER[0], max_zoom=SCALE_JITTER[1],
              mode=("bilinear", "nearest"), keep_size=False),
    # NOTE: pad befor ScaleIntensityRanged, worth fixing for new runs
    SpatialPadd(keys=["image", "label"], spatial_size=(CROP_SIZE, CROP_SIZE), mode="constant"),  # label pad 0 = ignore
    RandSpatialCropd(keys=["image", "label"], roi_size=(CROP_SIZE, CROP_SIZE), random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),  # hflip only 

    # Photometric augs 
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    RandScaleIntensityd(keys=["image"], factors=0.5, prob=0.5),                     
    RandShiftIntensityd(keys=["image"], offsets=0.125, prob=0.5),                   
    RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.5),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5, channel_wise=True),  
    NormalizeIntensityd(keys=["image"], subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    CastToTyped(keys=["image", "label"], dtype=(torch.float32, torch.int64)),
])

val_transforms = Compose([
    LoadImaged(
        keys=["image"],
        reader=PILReader(converter=lambda image: image.convert("RGB"), reverse_indexing=False),
    ),
    LoadADE20KLabeld(keys="label"),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["image"], subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
    CastToTyped(keys=["image", "label"], dtype=(torch.float32, torch.int64)),

    Resized(keys=["image", "label"], spatial_size=RESIZE_LONGEST, size_mode="longest", mode=("bilinear", "nearest")),
    DivisiblePadd(keys=["image", "label"], k=STRIDE_DIVISOR, mode="constant"),
])


val_transforms_fullres = Compose([
    LoadImaged(
        keys=["image"],
        reader=PILReader(converter=lambda image: image.convert("RGB"), reverse_indexing=False),
    ),
    LoadADE20KLabeld(keys="label"),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["image"], subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
    CastToTyped(keys=["image", "label"], dtype=(torch.float32, torch.int64)),
])

loader_kwargs = dict(num_workers=num_workers, pin_memory=False)
if num_workers > 0:
    loader_kwargs["prefetch_factor"] = 4

train_ds = CacheDataset(data=train_dataset, transform=train_transforms,
                        cache_rate=cache_rate, num_workers=num_workers)
train_loader = ThreadDataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                collate_fn=list_data_collate, **loader_kwargs)
# NOTE: MSPE Eval
val_ds = CacheDataset(data=val_dataset, transform=val_transforms,
                      cache_rate=cache_rate, num_workers=num_workers)
val_loader = ThreadDataLoader(val_ds, batch_size=1, shuffle=False, **loader_kwargs)

# NOTE: The Real eval
val_ds_fullres = CacheDataset(data=val_dataset, transform=val_transforms_fullres,
                              cache_rate=0.0, num_workers=num_workers)
val_loader_fullres = ThreadDataLoader(val_ds_fullres, batch_size=1, shuffle=False, **loader_kwargs)


In [ ]:
def _denormalize_for_display(image_chw):
    # Undo ImageNet normalization so previews look natural
    mean = torch.tensor(IMAGENET_MEAN).view(-1, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(-1, 1, 1)
    return (image_chw * std + mean).clamp(0, 1)


def display_sample_pairs(data_loader, n_samples=3, mask_alpha=0.5):
    fig, axes = plt.subplots(n_samples, 3, figsize=(18, 4 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for idx, batch in enumerate(data_loader):
        if idx >= n_samples:
            break

        image = _denormalize_for_display(batch["image"][0].cpu()).permute(1, 2, 0)
        mask = batch["label"][0][0].cpu()
        filename = os.path.basename(batch["image"].meta["filename_or_obj"][0])
        n_classes = len(torch.unique(mask))

        ax_row = axes[idx]
        ax_row[0].imshow(image)
        ax_row[0].set_title(f"Image\n{filename}", fontsize=9)
        ax_row[0].axis("off")

        ax_row[1].imshow(mask, cmap="tab20", interpolation="nearest")
        ax_row[1].set_title(f"ADE20K mask\n{n_classes} classes in image", fontsize=9)
        ax_row[1].axis("off")

        ax_row[2].imshow(image)
        ax_row[2].imshow(mask, cmap="tab20", alpha=mask_alpha, interpolation="nearest")
        ax_row[2].set_title(f"Image + mask overlay\n{filename}", fontsize=9)
        ax_row[2].axis("off")

    plt.tight_layout()
    plt.show()

# Take a look at train data
display_sample_pairs(train_loader, n_samples=3)


In [ ]:
# Fine-tuning configuration (frozen backbone, MSPE kernels only) -- R2 data/eval protocol

FEATURE_SIZE = 96
SWIN_DEPTHS = (2, 2, 6, 2)
SWIN_NUM_HEADS = (3, 6, 12, 24)
USE_V2 = False
ADE20K_NUM_CLASSES = 151
IGNORE_INDEX = 0  # ADE20K class 0

RECIPE_TAG = "R2"

MAX_EPOCHS = 10
VAL_INTERVAL = 2
EARLY_STOPPING_PATIENCE = 10  

LR = 1e-3  # only patch_embed trains here
WD = 1e-5
GRAD_CLIP = 1.0

# MSPE
MSPE_K = 3
ADE_RESOLUTIONS = [128, 256, 512]  
TRAIN_RESOLUTIONS = list(ADE_RESOLUTIONS)
TEST_RESOLUTIONS = [128, 256, 512, 640]

# Warm start each MSPE from the frozen baseline patch_embed
WARM_START = True

RESUME = False  # set True and rerun to continue from last_state_

DATASET_TAG = "ADE20K"
WANDB_PROJECT = "ADE20K_MSPE_SWIN_FINE_TUNING"

# local smoke-test overrides 
if not colab:
    MAX_EPOCHS = 2
    VAL_INTERVAL = 1

# Frozen baseline (SwinUNETR V1, patch size 4, stock patch embed)
BASELINE_CHECKPOINT = os.path.join(baseline_dir, f"best_metric_BASELINE_SWIN_V1_P4_{RECIPE_TAG}.pth")

# Save fine-tuned checkpoints 
if colab:
    CKPT_DIR = r"/content/drive/MyDrive/MSPE/NOTEBOOKS/ADE20K/FINE_TUNING/"
else:
    CKPT_DIR = os.getcwd()
os.makedirs(CKPT_DIR, exist_ok=True)

# MSPE variants to fine-tune (frozen backbone, only patch_embed trains)
EXPERIMENTS = [
    {
        "name": "FT_ROUTING_P4",
        "mspe_class": MSPEPatchEmbedSwinNaiveRouting,
        "description": "Frozen backbone + naive routing (K identical kernels)",
    },
    # {
    #     "name": "FT_OVERLAPPING_P4",
    #     "mspe_class": MSPEPatchEmbedSwinOverlapping,
    #     "description": "Frozen backbone + overlapping kernels",
    # },
    # {
    #     "name": "FT_DILATING_P4",
    #     "mspe_class": MSPEPatchEmbedSwinDilatingK3,
    #     "description": "Frozen backbone + dilating kernels",
    # },
]

# Run names and checkpoints 
for exp in EXPERIMENTS:
    exp["run_name"] = f"{exp['name']}_{RECIPE_TAG}"
    exp["checkpoint_path"] = os.path.join(CKPT_DIR, f"best_metric_{exp['name']}_{RECIPE_TAG}.pth")

print(f"Experiments to fine-tune: {[e['run_name'] for e in EXPERIMENTS]}")
print(f"Frozen baseline checkpoint: {BASELINE_CHECKPOINT}")
print(f"Backbone: feature_size={FEATURE_SIZE}, depths={SWIN_DEPTHS}, num_heads={SWIN_NUM_HEADS}, use_v2={USE_V2}")
print(f"Patch size: {PATCH_SIZE} (deepest stride / input divisor: {STRIDE_DIVISOR})")
print(f"MSPE resolutions: {ADE_RESOLUTIONS}, K: {MSPE_K}, warm-start kernels: {WARM_START}")
print(f"Test resolutions: {TEST_RESOLUTIONS}")
print(f"Epochs: {MAX_EPOCHS}, optimizer: AdamW, lr: {LR}, wd: {WD}, clip: {GRAD_CLIP}, scheduler: CosineAnnealingLR")
print(f"Train: base longest {BASE_LONGEST}, jitter {SCALE_JITTER}, crop {CROP_SIZE}x{CROP_SIZE}, batch: {batch_size}, cache_rate: {cache_rate}, workers: {num_workers}")
print(f"Val canvas: {RESIZE_LONGEST}-longest; native eval: sliding window {RESIZE_LONGEST}x{RESIZE_LONGEST}")
print(f"Saving fine-tuned checkpoints to: {CKPT_DIR}")


In [ ]:
# Resize helpers 

def get_aspect_preserving_target_size(inputs, effective_resolution, divisor=STRIDE_DIVISOR):
    spatial_shape = inputs.shape[2:]
    spatial_dims = len(spatial_shape)
    current_eff = float(np.prod(spatial_shape)) ** (1.0 / spatial_dims)
    scale = effective_resolution / current_eff

    target_size = []
    for dim in spatial_shape:
        resized_dim = int(round(dim * scale))
        resized_dim = max(divisor, int(round(resized_dim / divisor)) * divisor)
        target_size.append(resized_dim)
    return target_size

# Aspect-preserving resize of an eval batch to an effective resolution (None = keep as-is)
def resize_batch(inputs, labels, effective_resolution):
    if effective_resolution is None:
        return inputs, labels
    target_size = get_aspect_preserving_target_size(inputs, effective_resolution)
    return img_resize(inputs, target_size), label_resize(labels, target_size)

# Set WANDB_API_KEY in your environment (or Colab secrets) before running
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

def login_wandb():
    if colab:
        wandb.login(key=WANDB_API_KEY)


In [ ]:
# Warm-start helpers + model factories (frozen backbone, MSPE-only)

def create_patch_embed_baseline():
    """Stock MONAI 2D patch embedding (matches the trained _P4 baseline)."""
    return PatchEmbed(
        patch_size=PATCH_SIZE, in_chans=3, embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm, spatial_dims=2,
    )


def _load_baseline_backbone_only(model, baseline_checkpoint):
    """Load every baseline weight EXCEPT swinViT.patch_embed.* (MSPE embed starts fresh)."""
    checkpoint = torch.load(baseline_checkpoint, weights_only=True)

    backbone_state = {
        name: tensor for name, tensor in checkpoint.items()
        if not name.startswith("swinViT.patch_embed.")
    }
    skipped_patch_keys = len(checkpoint) - len(backbone_state)

    incompatible = model.load_state_dict(backbone_state, strict=False)
    unexpected_keys = list(incompatible.unexpected_keys)
    missing_non_patch_keys = [
        name for name in incompatible.missing_keys
        if not name.startswith("swinViT.patch_embed.")
    ]
    if unexpected_keys or missing_non_patch_keys:
        raise RuntimeError(
            "Backbone-only checkpoint load failed. "
            f"Unexpected keys: {unexpected_keys}; missing non-patch keys: {missing_non_patch_keys}"
        )
    print(f"Loaded baseline backbone from {baseline_checkpoint}")
    print(f"Skipped baseline patch_embed keys: {skipped_patch_keys}")


def _warm_start_mspe_from_baseline(mspe_embed, baseline_checkpoint, device):
    """Initialise every MSPE kernel from the pretrained baseline patch_embed conv (4x4)."""
    if not hasattr(mspe_embed, "patch_kernels"):
        print("Embedding has no patch_kernels; skipping warm-start.")
        return

    checkpoint = torch.load(baseline_checkpoint, weights_only=True)
    proj_w = checkpoint["swinViT.patch_embed.proj.weight"].to(device) 
    proj_b = checkpoint["swinViT.patch_embed.proj.bias"].to(device)
    base_size = tuple(proj_w.shape[2:])

    with torch.no_grad():
        for k in range(len(mspe_embed.patch_kernels)):
            kernel = mspe_embed.patch_kernels[k]
            k_size = tuple(kernel.weight.shape[2:])
            if k_size == base_size:
                kernel.weight.copy_(proj_w)                                      # exact 
            else:
                kernel.weight.copy_(pi_resize(proj_w, list(k_size)).to(device))  # PI-resize to k_size
            if kernel.bias is not None:
                kernel.bias.copy_(proj_b)
        # warm start the embedding norm affine
        if mspe_embed.norm is not None and "swinViT.patch_embed.norm.weight" in checkpoint:
            mspe_embed.norm.weight.copy_(checkpoint["swinViT.patch_embed.norm.weight"].to(device))
            mspe_embed.norm.bias.copy_(checkpoint["swinViT.patch_embed.norm.bias"].to(device))

    print(f"Warm-started {len(mspe_embed.patch_kernels)} MSPE kernels from baseline patch_embed "
          f"(exact copy where kernel size == {base_size}, PI-resize otherwise).")


def _create_swinunetr():
    model = monai.networks.nets.SwinUNETR(
        in_channels=3, out_channels=ADE20K_NUM_CLASSES, spatial_dims=2,
        patch_size=PATCH_SIZE,
        feature_size=FEATURE_SIZE, depths=SWIN_DEPTHS, num_heads=SWIN_NUM_HEADS,
        use_v2=USE_V2,
    )

    # Fix for default MONNAI patch_size = 2
    model.decoder1 = UnetrUpBlock(
        spatial_dims=2,
        in_channels=FEATURE_SIZE,
        out_channels=FEATURE_SIZE,
        kernel_size=3,
        upsample_kernel_size=PATCH_SIZE,
        norm_name="instance",
        res_block=True,
    )

    def _check_input_size(spatial_shape):
        if any(s % STRIDE_DIVISOR for s in spatial_shape):
            raise ValueError(
                f"spatial shape {tuple(spatial_shape)} must be divisible by {STRIDE_DIVISOR}"
            )
    model._check_input_size = _check_input_size

    return model.to(device)


def create_baseline_model(baseline_checkpoint, device):
    """Rebuild the stock-embed baseline and load the full checkpoint (for reference eval)."""
    model = _create_swinunetr()
    model.swinViT.patch_embed = create_patch_embed_baseline().to(device)
    model.load_state_dict(torch.load(baseline_checkpoint, weights_only=True))
    model.eval()
    print(f"Loaded baseline checkpoint for evaluation from {baseline_checkpoint}")
    return model


def create_model_for_variant(variant_class, baseline_checkpoint, device):
    """SwinUNETR with a frozen baseline backbone + a trainable (warm-started) MSPE patch embed."""
    model = _create_swinunetr()

    # Load only the frozen backbone -- skip the baseline patch_embed
    _load_baseline_backbone_only(model, baseline_checkpoint)

    # Build + swap the MSPE patch embedding
    mspe_embed = variant_class(
        patch_size=PATCH_SIZE, in_chans=3, embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm, spatial_dims=2,
        K=MSPE_K, resolutions=ADE_RESOLUTIONS,
    ).to(device)

    if WARM_START:
        _warm_start_mspe_from_baseline(mspe_embed, baseline_checkpoint, device)

    model.swinViT.patch_embed = mspe_embed
    print(f"Swapped patch_embed to: {variant_class.__name__}")
    print(model.swinViT.patch_embed)

    # Freeze everything except the patch embedding
    for name, param in model.named_parameters():
        if "patch_embed" not in name:
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    frozen_params = total_params - trainable_params
    print(f" Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f" Frozen params:    {frozen_params:,} ({100 * frozen_params / total_params:.2f}%)")

    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"  Trainable parameter groups ({len(trainable_names)}):")
    for n in trainable_names:
        print(f"   - {n}")

    return model


In [ ]:
# Confusion-matrix metrics (aAcc/mAcc/mIoU/macro-F1) 

# Accumulate a full confusion matrix
def update_confmat(cm, logits, labels):
    preds = logits.argmax(dim=1).reshape(-1)  
    gts = labels[:, 0].reshape(-1).long()
    valid = gts != IGNORE_INDEX
    idx = gts[valid] * ADE20K_NUM_CLASSES + preds[valid]
    binc = torch.bincount(idx, minlength=ADE20K_NUM_CLASSES ** 2)
    return cm + binc.reshape(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES)

def metrics_from_confmat(cm):
    cm = cm.double()
    tp = torch.diag(cm)
    gt = cm.sum(dim=1)  # row totals
    pred = cm.sum(dim=0)
    fn = gt - tp
    fp = pred - tp
    sel = slice(1, ADE20K_NUM_CLASSES)
    iou = (tp / (tp + fp + fn))[sel]
    acc = (tp / gt)[sel]
    f1 = (2 * tp / (2 * tp + fp + fn))[sel]
    total = gt[sel].sum()
    return {
        "mean_iou": iou.nanmean().item(),
        "mean_acc": acc.nanmean().item(),
        "mean_f1": f1.nanmean().item(),
        "aacc": (tp[sel].sum() / total).item() if total > 0 else float("nan"),
        "iou_vals": iou.cpu(),
        "acc_vals": acc.cpu(),
        "f1_vals": f1.cpu(),
    }

# Print a summary
def print_metric_table(title, result, worst_k=5):
    print(title)
    print(f"  aAcc: {result['aacc']:.4f} | mAcc: {result['mean_acc']:.4f} | "
          f"mIoU: {result['mean_iou']:.4f} | macro-F1: {result['mean_f1']:.4f}")
    iou_vals = result["iou_vals"]
    valid = ~torch.isnan(iou_vals)
    if valid.any():
        ranked = torch.where(valid, iou_vals, torch.full_like(iou_vals, float("inf")))
        order = torch.argsort(ranked)[:worst_k]
        print(f"  worst {worst_k} classes by IoU:")
        for i in order.tolist():
            print(f"    class {i + 1:>3} | IoU {iou_vals[i]:.4f} | "
                  f"Acc {result['acc_vals'][i]:.4f} | F1 {result['f1_vals'][i]:.4f}")
    print()
    return result

# Log to wandb 
def log_eval_result(prefix, result, best_metric_epoch):
    if wandb.run is None:
        return

    log_data = {
        f"{prefix}/mIoU": result["mean_iou"],
        f"{prefix}/mAcc": result["mean_acc"],
        f"{prefix}/macroF1": result["mean_f1"],
        f"{prefix}/aAcc": result["aacc"],
        "best_metric_epoch": best_metric_epoch,
    }
    for c, (iou_v, acc_v, f1_v) in enumerate(
        zip(result["iou_vals"], result["acc_vals"], result["f1_vals"]), start=1
    ):
        log_data[f"{prefix}/class_{c}_iou"] = iou_v.item()
        log_data[f"{prefix}/class_{c}_acc"] = acc_v.item()
        log_data[f"{prefix}/class_{c}_f1"] = f1_v.item()
    wandb.log(log_data)

# Evals robustness only (buggy code :))
def evaluate_validation_loader(model, data_loader, resolution=None, prefix="validation_native", best_metric_epoch=-1):
    cm = torch.zeros(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES, dtype=torch.long, device=device)

    with torch.no_grad():
        for val_data in data_loader:
            val_inputs, val_labels = (
                val_data["image"].to(device, non_blocking=True),
                val_data["label"].to(device, non_blocking=True),
            )
            val_inputs, val_labels = resize_batch(val_inputs, val_labels, resolution)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                val_outputs = model(val_inputs)

            cm = update_confmat(cm, val_outputs, val_labels)
            del val_outputs, val_inputs, val_labels

    result = metrics_from_confmat(cm)
    title = "\nNative validation metrics" if resolution is None else f"Resized validation metrics at effective res {resolution}"
    print_metric_table(title, result)
    log_eval_result(prefix, result, best_metric_epoch)
    return result

# Real eval
def evaluate_native_sliding_window(model, fullres_loader, roi_size=(RESIZE_LONGEST, RESIZE_LONGEST),
                                   sw_batch_size=4, overlap=0.5,
                                   prefix="validation_native_sw", best_metric_epoch=-1):
    cm = torch.zeros(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES, dtype=torch.long, device=device)

    def _predict(patch):
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            return model(patch)

    with torch.no_grad():
        for val_data in fullres_loader:
            val_inputs = val_data["image"].to(device, non_blocking=True)
            val_labels = val_data["label"].to(device, non_blocking=True)
            val_outputs = sliding_window_inference(
                val_inputs, roi_size=roi_size, sw_batch_size=sw_batch_size,
                predictor=_predict, overlap=overlap, mode="gaussian",
            )
            cm = update_confmat(cm, val_outputs, val_labels)
            del val_outputs, val_inputs, val_labels

    result = metrics_from_confmat(cm)
    print_metric_table("\nNative validation metrics (sliding window)", result)
    log_eval_result(prefix, result, best_metric_epoch)
    return result

# Compute val evals
def evaluate_validation_all_resolutions(model, val_loader, val_loader_fullres, exp_name, best_metric_epoch,
                                        checkpoint_path, roi_size=(RESIZE_LONGEST, RESIZE_LONGEST), sw_overlap=0.5):
    print(f"\n{'=' * 40}")
    print(f"VALIDATION EVALUATION: {exp_name}")
    print(f"{'=' * 40}")

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.eval()

    native_result = evaluate_native_sliding_window(
        model, val_loader_fullres, roi_size=roi_size, overlap=sw_overlap,
        prefix="validation_native_sw", best_metric_epoch=best_metric_epoch,
    )
    torch.cuda.empty_cache() 
    gc.collect()

    # Downsampled effective resolutions: keep the single-pass model(x) evaluation.
    multi_resolution_results = {}
    for hw in TEST_RESOLUTIONS:
        multi_resolution_results[hw] = evaluate_validation_loader(
            model, val_loader, resolution=hw,
            prefix=f"validation_res_{hw}", best_metric_epoch=best_metric_epoch,
        )

    if wandb.run is not None:
        wandb.log({
            "validation/mIoU": native_result["mean_iou"],
            "validation/mAcc": native_result["mean_acc"],
            "validation/macroF1": native_result["mean_f1"],
            "validation/aAcc": native_result["aacc"],
            "best_metric_epoch": best_metric_epoch,
        })

    return {"native": native_result, "multi_res": multi_resolution_results}


# Selection metric during training
def run_validation_selection(model, val_loader):
    sel_resolutions = [None] + TRAIN_RESOLUTIONS  
    cms = {
        r: torch.zeros(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES, dtype=torch.long, device=device)
        for r in sel_resolutions
    }
    model.eval()
    with torch.no_grad():
        for val_data in val_loader:  # single loader pass, every resolution per batch
            val_inputs = val_data["image"].to(device, non_blocking=True)
            val_labels = val_data["label"].to(device, non_blocking=True)
            for r in sel_resolutions:
                inputs_r, labels_r = resize_batch(val_inputs, val_labels, r)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    outputs_r = model(inputs_r)
                cms[r] = update_confmat(cms[r], outputs_r, labels_r)

    per_res = {
        ("canvas" if r is None else r): metrics_from_confmat(cms[r])["mean_iou"]
        for r in sel_resolutions
    }
    mean_miou = float(np.mean(list(per_res.values())))
    return mean_miou, per_res


In [ ]:
# MSPE kernel drift analysis

def analyze_kernel_drift(initial_state, model, exp_name):
    patch_embed = model.swinViT.patch_embed
    if not hasattr(patch_embed, "patch_kernels"):
        print(f"{exp_name}: no MSPE kernels; skipping drift analysis.")
        return None

    final_state = patch_embed.state_dict()
    metrics_l2, metrics_cos, metrics_max = [], [], []

    print(f"\nKernel drift analysis: {exp_name}")
    print("Kernel | L2 Dist  | Cosine Sim | Max Diff")
    print("-" * 44)
    for k in range(len(patch_embed.patch_kernels)):
        key = f"patch_kernels.{k}.weight"
        if key not in initial_state or key not in final_state:
            raise KeyError(f"Missing MSPE weight key for drift analysis: {key}")
        w_init = initial_state[key].detach().flatten().cpu()
        w_final = final_state[key].detach().flatten().cpu()
        l2 = torch.norm(w_final - w_init, p=2).item()
        cos = F.cosine_similarity(w_final.unsqueeze(0), w_init.unsqueeze(0)).item()
        mx = torch.max(torch.abs(w_final - w_init)).item()
        metrics_l2.append(l2); metrics_cos.append(cos); metrics_max.append(mx)
        print(f"{k:>6} | {l2:>8.4f} | {cos:>10.4f} | {mx:>8.4f}")

    drift = {"l2": metrics_l2, "cosine": metrics_cos, "max_diff": metrics_max}

    if wandb.run is not None:
        log_data = {}
        for k, (l2, cos, mx) in enumerate(zip(metrics_l2, metrics_cos, metrics_max)):
            log_data[f"kernel_drift/kernel_{k}_l2"] = l2
            log_data[f"kernel_drift/kernel_{k}_cosine"] = cos
            log_data[f"kernel_drift/kernel_{k}_max_diff"] = mx
        wandb.log(log_data)

    x = np.arange(len(metrics_l2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(x, metrics_l2); axes[0].set_title("L2 distance"); axes[0].set_xlabel("Kernel")
    axes[1].bar(x, metrics_cos); axes[1].set_title("Cosine similarity"); axes[1].set_xlabel("Kernel")
    axes[2].bar(x, metrics_max); axes[2].set_title("Max absolute diff"); axes[2].set_xlabel("Kernel")
    fig.suptitle(f"MSPE kernel drift: {exp_name}")
    plt.tight_layout()

    drift_path = os.path.join(root_dir, f"kernel_drift_{exp_name}.png")
    fig.savefig(drift_path, dpi=150, bbox_inches="tight")
    if wandb.run is not None:
        wandb.log({"kernel_drift/plot": wandb.Image(fig)})
    plt.show()
    return drift


In [ ]:
# Training loop (frozen backbone, only the MSPE patch_embed trains)


def mspe_swin_train_step_accum(
    model, img, label, loss_fn, scaler,
    lam=1.0, amp_dtype=AMP_DTYPE, device_type=device.type,
):
    patch_embed = model.swinViT.patch_embed
    hw_list = patch_embed.sample_resolutions()  # K resolutions, one per kernel
    K = len(hw_list)
    norm = K + 1
    running_loss = 0.0

    def _forward_backward(fwd_img, fwd_label, func_idx, weight):
        nonlocal running_loss
        with torch.autocast(device_type=device_type, dtype=amp_dtype):
            if func_idx is None:
                logits = model(fwd_img)
            else:
                logits = mspe_swin_forward(model, fwd_img, func_idx=func_idx)
            loss = weight * loss_fn(logits, fwd_label) / norm
        scaler.scale(loss).backward()
        running_loss += loss.item()

    # K resolution-specific forwards (each backproped immediately)
    for k, eff_target_k in enumerate(hw_list):
        target_size = get_aspect_preserving_target_size(img, eff_target_k)
        img_k = img_resize(img, target_size)
        label_k = label_resize(label, target_size)
        _forward_backward(img_k, label_k, func_idx=k, weight=1.0)
        del img_k, label_k

    # Native crop forward 
    _forward_backward(img, label, func_idx=None, weight=lam)
    return running_loss


def train_variant(model, train_loader, val_loader, exp_config):
    exp_name = exp_config["name"]
    checkpoint_path = exp_config["checkpoint_path"]
    # For resume after a runtime disconnect
    last_state_path = checkpoint_path.replace("best_metric_", "last_state_")

    # Background is ignored
    ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    def loss_function(logits, targets):
        return ce_loss(logits, targets[:, 0].long())

    # Optimize ONLY the patch_embed
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WD)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler = GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

    best_metric = -1  # best mean selection mIoU
    best_metric_epoch = -1
    epoch_loss_values, metric_values = [], []
    epochs_no_improve = 0
    completed_epochs = 0
    start_epoch = 0

    # Resume insurance 
    if RESUME and os.path.exists(last_state_path):
        state = torch.load(last_state_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch = state["epoch"]
        best_metric = state["best_metric"]
        best_metric_epoch = state["best_metric_epoch"]
        epochs_no_improve = state["epochs_no_improve"]
        epoch_loss_values = state["epoch_loss_values"]
        metric_values = state["metric_values"]
        completed_epochs = start_epoch
        print(f"Resumed training state from {last_state_path} at epoch {start_epoch}")

    total_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_start = time.time()
        print("-" * 10)
        print(f"{exp_name}: epoch {epoch + 1}/{MAX_EPOCHS}")
        model.train()
        epoch_loss = 0
        step = 0

        for batch_data in train_loader:
            step += 1
            inputs = batch_data["image"].to(device, non_blocking=True)
            labels = batch_data["label"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            # MSPE step with per-forward backward
            step_loss = mspe_swin_train_step_accum(
                model, inputs, labels, loss_function, scaler, lam=1.0,
            )
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += step_loss
            if wandb.run is not None:
                wandb.log({
                    "train/step_loss": step_loss,
                    "train/amp_scale": scaler.get_scale(),
                    "train/grad_norm": grad_norm.item(),
                })
            if step % 5 == 0 or step == len(train_loader):  # cleaner stdout
                print(f"{step}/{len(train_loader)}, train_loss: {step_loss:.4f}")

        epoch_loss /= step
        epoch_loss_values.append(epoch_loss)
        completed_epochs = epoch + 1
        epoch_time_sec = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
        print(f"time per epoch: {epoch_time_sec:.2f} s")

        if wandb.run is not None:
            wandb.log({
                "train/epoch_loss": epoch_loss, "lr": current_lr,
                "epoch": epoch + 1, "train/time_per_epoch": epoch_time_sec,
            })

        if (epoch + 1) % VAL_INTERVAL == 0:
            metric, per_res = run_validation_selection(model, val_loader)
            metric_values.append(metric)
            per_res_str = ", ".join(f"{k}: {v:.4f}" for k, v in per_res.items())
            print(f"selection mean mIoU: {metric:.4f} ({per_res_str})")

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_path)
                print(f"saved new best metric model to {checkpoint_path}")
                print(f"best selection mean mIoU: {best_metric:.4f} at epoch: {best_metric_epoch}")
            else:
                epochs_no_improve += VAL_INTERVAL

            if wandb.run is not None:
                log_data = {"val mean mIoU": metric, "epoch": epoch + 1}
                for k, v in per_res.items():
                    log_data[f"val mIoU res_{k}"] = v
                wandb.log(log_data)

            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch + 1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs).")
                if wandb.run is not None:
                    wandb.log({"early_stop_epoch": epoch + 1})
                break

        scheduler.step()

        # Resume insurance every 
        if (epoch + 1) % 10 == 0:
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": epoch + 1,
                "best_metric": best_metric,
                "best_metric_epoch": best_metric_epoch,
                "epochs_no_improve": epochs_no_improve,
                "epoch_loss_values": epoch_loss_values,
                "metric_values": metric_values,
            }, last_state_path)

        gc.collect()

    total_time = time.time() - total_start
    avg_epoch_time = total_time / max(completed_epochs, 1)
    print(f"total epochs: {completed_epochs}, total training time: {total_time / 60:.2f} min, "
          f"average time per epoch: {avg_epoch_time:.2f} s")
    print(f"train completed, best selection mean mIoU: {best_metric:.4f} at epoch: {best_metric_epoch}")
    return best_metric, best_metric_epoch, epoch_loss_values, metric_values


In [ ]:
# Baseline checkpoint eval -- native sliding-window + multi-resolution

login_wandb()

# Iterrupted execution fix
if wandb.run is not None:
    wandb.finish()

    wandb.init(
    project=WANDB_PROJECT,
    name=f"BASELINE_CHECKPOINT_EVAL_P4_{RECIPE_TAG}",
    tags=[DATASET_TAG, RECIPE_TAG, "baseline_eval"],
    mode=None if colab else "disabled",  
    config={
        "recipe": RECIPE_TAG,
        "checkpoint": BASELINE_CHECKPOINT,
        "feature_size": FEATURE_SIZE,
        "patch_size": PATCH_SIZE,
        "num_classes": ADE20K_NUM_CLASSES,
        "eval_resolutions": TEST_RESOLUTIONS,
        "eval_only": True,
        "use_v2": USE_V2,
        "native_eval": "sliding_window",
        "sw_roi_size": RESIZE_LONGEST,
        "sw_overlap": 0.5,
    },
)

try:
    baseline_model = create_baseline_model(BASELINE_CHECKPOINT, device)

    baseline_results = evaluate_validation_all_resolutions(
        baseline_model, val_loader, val_loader_fullres, "BASELINE", -1,
        checkpoint_path=BASELINE_CHECKPOINT,
    )
finally:
    if wandb.run is not None:
        wandb.finish()

del baseline_model
torch.cuda.empty_cache()
gc.collect()
print("Baseline checkpoint evaluation complete.")


In [ ]:
# Fine-tuning experiment loop

login_wandb()
assert "baseline_results" in globals(), (
    "Run the baseline checkpoint eval cell before running the experiments loop."
)

all_final_results = {"BASELINE": baseline_results}
all_training_curves = {}
all_drift = {}

for exp_idx, exp in enumerate(EXPERIMENTS):
    set_determinism(seed=2026)
    torch.backends.cudnn.benchmark = True
    torch.use_deterministic_algorithms(False)

    exp_name = exp["name"]
    print(f"\n{'=' * 60}\nEXPERIMENT {exp_idx + 1}/{len(EXPERIMENTS)}: {exp['run_name']}\n{exp['description']}\n{'=' * 60}\n")

    if wandb.run is not None:
        wandb.finish()

    wandb.init(
        project=WANDB_PROJECT,
        name=exp["run_name"],
        save_code=True,
        group=exp_name,
        tags=[DATASET_TAG, RECIPE_TAG],
        mode=None if colab else "disabled",   
        config={
            "recipe": RECIPE_TAG,
            "max_epochs": MAX_EPOCHS,
            "val_interval": VAL_INTERVAL,
            "batch_size": batch_size,
            "feature_size": FEATURE_SIZE,
            "patch_size": PATCH_SIZE,
            "depths": SWIN_DEPTHS,
            "num_heads": SWIN_NUM_HEADS,
            "use_v2": USE_V2,
            "num_classes": ADE20K_NUM_CLASSES,
            "training_strategy": "frozen_backbone_mspe_only",
            "mspe_variant": exp["mspe_class"].__name__,
            "mspe_resolutions": ADE_RESOLUTIONS,
            "mspe_K": MSPE_K,
            "train_resolutions": TRAIN_RESOLUTIONS,
            "test_resolutions": TEST_RESOLUTIONS,
            "selection_metric": "mean mIoU over canvas + train resolutions",
            "frozen_backbone": True,
            "warm_start_kernels": WARM_START,
            "baseline_checkpoint": BASELINE_CHECKPOINT,
            "amp_dtype": str(AMP_DTYPE),
            "learning_rate": LR,
            "weight_decay": WD,
            "grad_clip": GRAD_CLIP,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "loss_type": "CrossEntropy",
            "ignore_index": IGNORE_INDEX,
            "base_longest": BASE_LONGEST,
            "crop_size": CROP_SIZE,
            "scale_jitter": list(SCALE_JITTER),
            "hflip_prob": 0.5,
            "resize_longest": RESIZE_LONGEST,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "native_eval": "sliding_window",
            "sw_roi_size": RESIZE_LONGEST,
            "sw_overlap": 0.5,
            "dataset": DATASET_TAG,
        },
    )

    try:
        # Frozen model with a warm started MSPE patch embedding
        model = create_model_for_variant(exp["mspe_class"], BASELINE_CHECKPOINT, device)

        # MSPE state AFTER warm-start for drift
        initial_mspe_state = copy.deepcopy(model.swinViT.patch_embed.state_dict())

        best_metric, best_metric_epoch, epoch_losses, metric_vals = train_variant(model, train_loader, val_loader, exp)
        all_training_curves[exp_name] = {
            "epoch_losses": epoch_losses,
            "metric_values": metric_vals,
            "best_metric": best_metric,
            "best_metric_epoch": best_metric_epoch,
            "checkpoint_path": exp["checkpoint_path"],
        }

        all_final_results[exp_name] = evaluate_validation_all_resolutions(
            model, val_loader, val_loader_fullres, exp_name, best_metric_epoch,
            checkpoint_path=exp["checkpoint_path"],
        )

        all_drift[exp_name] = analyze_kernel_drift(initial_mspe_state, model, exp_name)
    finally:
        if wandb.run is not None:
            wandb.finish()

    del model
    torch.cuda.empty_cache()
    print(f"\n--- Completed {exp_name}")

print("\n" + "=" * 60)
print("ALL FINE-TUNING EXPERIMENTS COMPLETED")
print("=" * 60)


In [ ]:
# Summary tables

print(f"\n{'=' * 80}")
print("VALIDATION RESULTS SUMMARY: MSPE fine-tuning (frozen backbone)")
print(f"{'=' * 80}\n")

res_cols = ["Native(SW)"] + [str(r) for r in TEST_RESOLUTIONS]
header = f"{'Condition':<26} | " + " | ".join(f"{c:>10}" for c in res_cols) + " |"
sep = "-" * len(header)

# One table per metric across native + test resolutions
metric_specs = [
    ("mIoU", "mean_iou"),
    ("mAcc", "mean_acc"),
    ("macro-F1", "mean_f1"),
    ("aAcc", "aacc"),
]
for label, key in metric_specs:
    print(f"\nValidation {label} (classes 1-150):")
    print(sep)
    print(header)
    print(sep)
    for exp_name, results in all_final_results.items():
        row = f"{exp_name:<26} | {results['native'][key]:>10.4f}"
        for hw in TEST_RESOLUTIONS:
            row += f" | {results['multi_res'][hw][key]:>10.4f}"
        row += " |"
        print(row)
    print(sep)

print("\nTraining Summary:")
print(f"{'Condition':<26} | {'Best selIoU':>11} | {'Best Epoch':>10} | {'Final Loss':>10}")
print("-" * 69)
for exp_name, curves in all_training_curves.items():
    final_loss = curves["epoch_losses"][-1] if curves["epoch_losses"] else float("nan")
    print(f"{exp_name:<26} | {curves['best_metric']:>11.4f} | {curves['best_metric_epoch']:>10d} | {final_loss:>10.4f}")

if all_drift:
    print("\nMSPE Kernel Drift Summary:")
    drift_header = f"{'Condition':<26} | {'Kernel':>6} | {'L2':>8} | {'Cosine':>10} | {'Max Diff':>8}"
    print(drift_header)
    print("-" * len(drift_header))
    for exp_name, drift in all_drift.items():
        if drift is None:
            continue
        for k, (l2, cos, mx) in enumerate(zip(drift["l2"], drift["cosine"], drift["max_diff"])):
            print(f"{exp_name:<26} | {k:>6d} | {l2:>8.4f} | {cos:>10.4f} | {mx:>8.4f}")


In [ ]:
# Training curves + resolution robustness

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

ax = axes[0]
for exp_name, curves in all_training_curves.items():
    epochs = list(range(1, len(curves["epoch_losses"]) + 1))
    ax.plot(epochs, curves["epoch_losses"], label=exp_name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for exp_name, curves in all_training_curves.items():
    val_epochs = [VAL_INTERVAL * (i + 1) for i in range(len(curves["metric_values"]))]
    ax.plot(val_epochs, curves["metric_values"], label=exp_name, marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Selection mean mIoU")
ax.set_title("Validation selection metric")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[2]
res_cols = ["Native(SW)"] + [str(r) for r in TEST_RESOLUTIONS]
res_x = list(range(len(res_cols)))
for exp_name, results in all_final_results.items():
    miou_curve = [results["native"]["mean_iou"]] + [results["multi_res"][r]["mean_iou"] for r in TEST_RESOLUTIONS]
    ax.plot(res_x, miou_curve, label=exp_name, marker="o", markersize=4)
ax.set_xticks(res_x)
ax.set_xticklabels(res_cols, rotation=45)
ax.set_xlabel("Effective resolution")
ax.set_ylabel("mIoU")
ax.set_title("Resolution robustness")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(root_dir, "fine_tuning_curves.png")
fig.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Viz and evals

def to_display_image(tensor_chw):
    image = tensor_chw.detach().cpu()
    if image.shape[0] in (3, 4):
        return _denormalize_for_display(image[:3]).permute(1, 2, 0)
    return image[0]


vis_resolutions = [None] + list(TEST_RESOLUTIONS)

login_wandb()

for exp in EXPERIMENTS:
    ckpt = exp["checkpoint_path"]
    print(f"Visualizing {exp['name']} from {ckpt}")

    model_viz = create_model_for_variant(exp["mspe_class"], BASELINE_CHECKPOINT, device)
    model_viz.load_state_dict(torch.load(ckpt, weights_only=True))
    model_viz.eval()

    with torch.no_grad():
        val_data = first(val_loader)
        val_inputs = val_data["image"].to(device, non_blocking=True)
        val_labels = val_data["label"].to(device, non_blocking=True)

        n_cols = len(vis_resolutions)
        fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 12))
        if n_cols == 1:
            axes = axes.reshape(3, 1)

        for col, effective_resolution in enumerate(vis_resolutions):
            inputs_r, labels_r = resize_batch(val_inputs, val_labels, effective_resolution)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                outputs = model_viz(inputs_r)

            image = to_display_image(inputs_r[0])
            label = labels_r[0, 0].detach().cpu()
            pred = torch.argmax(outputs, dim=1).detach().cpu()[0]

            h, w = inputs_r.shape[-2], inputs_r.shape[-1]
            col_title = f"canvas\n{h}x{w}" if effective_resolution is None else f"eff {effective_resolution}\n{h}x{w}"

            axes[0, col].imshow(image)
            axes[0, col].set_title(col_title)
            axes[0, col].axis("off")
            axes[1, col].imshow(label, cmap="viridis")
            axes[1, col].set_title("label")
            axes[1, col].axis("off")
            axes[2, col].imshow(pred, cmap="viridis")
            axes[2, col].set_title("prediction")
            axes[2, col].axis("off")

        fig.suptitle(exp["name"], fontsize=14)
        plt.tight_layout()

        viz_path = os.path.join(root_dir, f"viz_{exp['run_name']}.png")
        fig.savefig(viz_path, dpi=300, bbox_inches="tight")
        print(f"saved visualization to {viz_path}")

        # Save inference image to wandb
        wandb.init(
            project=WANDB_PROJECT,
            name=f"{exp['run_name']}_viz",
            group=exp["name"],
            tags=[DATASET_TAG, RECIPE_TAG, "viz"],
            mode=None if colab else "disabled",
        )
        wandb.log({"predictions": wandb.Image(fig)})
        wandb.finish()

        plt.show()
        plt.close(fig)

    del model_viz
    torch.cuda.empty_cache()


In [ ]:
# Disconnect from runtime
runtime.unassign()
